**Introduction**

    In this project, we will be primarily working on the IMDB reviews given by the viewers about various movies.

    We will be getting the dataset from the tensorflow_datasets. Then we will develop an understanding of how to preprocess the data, convert the English words to numerical representations, and prepare it to be fed as input for our deep learning model with GRUs.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
tf.random.set_seed(42)

In [2]:
#Loading Dataset
datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)

In [3]:
print(datasets.keys())

dict_keys(['test', 'train', 'unsupervised'])


In [4]:
train_size = info.splits["train"].num_examples
test_size = info.splits["test"].num_examples

print(train_size , test_size)

25000 25000


In [5]:
#datasets["train"].batch(2) batches 2 data samples at a time.
#datasets["train"].batch(2).take(1) allows to take 1 batch at a time.
for X_batch, y_batch in datasets["train"].batch(2).take(1):
    for review, label in zip(X_batch.numpy(), y_batch.numpy()):
        print("Review:", review.decode("utf-8")[:200], "...")
        print("Label:", label, "= Positive" if label else "= Negative")
        print()

Review: This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting  ...
Label: 0 = Negative

Review: I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However  ...
Label: 0 = Negative



tf.strings - Operations for working with string Tensors.

tf.strings.substr(X_batch, 0, 300) - For each string in the input Tensor X_batch, it creates a substring starting at index pos(here 0) with a total length of len(here 300). So basically, it returns substrings from Tensor of strings.

tf.strings.regex_replace(X_batch, rb"<br\s*/?>", b" ") - Replaces elements of X_batch matching regex pattern <br\s*/?> with rewrite .

tf.strings.split(X_batch) - Split elements of input X_batch into a RaggedTensor.

X_batch.to_tensor(default_value=b"<pad>") - Converts the RaggedTensor into a tf.Tensor. default_value is the value to set for indices not specified in X_batch. Empty values are assigned default_value(here <pad>).

In [6]:
def preprocess(X_batch, y_batch):
    X_batch = tf.strings.substr(X_batch, 0, 300)
    X_batch = tf.strings.regex_replace(X_batch, rb"<br\s*/?>", b" ")
    X_batch = tf.strings.regex_replace(X_batch, b"[^a-zA-Z']", b" ")
    X_batch = tf.strings.split(X_batch)
    return X_batch.to_tensor(default_value=b"<pad>"), y_batch

In [7]:
preprocess(X_batch, y_batch)

(<tf.Tensor: shape=(2, 53), dtype=string, numpy=
 array([[b'This', b'was', b'an', b'absolutely', b'terrible', b'movie',
         b"Don't", b'be', b'lured', b'in', b'by', b'Christopher',
         b'Walken', b'or', b'Michael', b'Ironside', b'Both', b'are',
         b'great', b'actors', b'but', b'this', b'must', b'simply', b'be',
         b'their', b'worst', b'role', b'in', b'history', b'Even',
         b'their', b'great', b'acting', b'could', b'not', b'redeem',
         b'this', b"movie's", b'ridiculous', b'storyline', b'This',
         b'movie', b'is', b'an', b'early', b'nineties', b'US',
         b'propaganda', b'pi', b'<pad>', b'<pad>', b'<pad>'],
        [b'I', b'have', b'been', b'known', b'to', b'fall', b'asleep',
         b'during', b'films', b'but', b'this', b'is', b'usually', b'due',
         b'to', b'a', b'combination', b'of', b'things', b'including',
         b'really', b'tired', b'being', b'warm', b'and', b'comfortable',
         b'on', b'the', b'sette', b'and', b'having', b'j

In [8]:
from collections import Counter

In [9]:
vocabulary = Counter()

In [10]:
for X_batch, y_batch in datasets["train"].batch(2).map(preprocess):
    for review in X_batch:
        vocabulary.update(list(review.numpy()))

In [11]:
#There are more than 50,000 words in the vocabulary. So let us truncate it to have only 10,000 most common words.
vocab_size = 10000

In [12]:
#Extract the top 10,000 most frequently occurring words from vocabulary and store these words in truncated_vocabulary list
truncated_vocabulary = [ word for word, count in vocabulary.most_common()[:vocab_size]]

**Creating a lookup table**
    
    Computer can only process numbers but not words. Thus we need to convert the words in truncated_vocabulary into numbers.

    So we now need to add a preprocessing step to replace each word with its ID (i.e., its index in the truncated_vocabulary). We will create a lookup table for this, using 1,000 out-of-vocabulary (oov) buckets.

    We shall create the lookup table such that the most frequently occurring words have lower indices than less frequently occurring words.

In [13]:
#tensor words containing the words of truncated_vocabulary.
words = tf.constant(truncated_vocabulary)

In [14]:
word_ids = tf.range(len(truncated_vocabulary), dtype=tf.int64)

In [15]:
vocab_init = tf.lookup.KeyValueTensorInitializer(words, word_ids)

In [17]:
num_oov_buckets = 1000
table = tf.lookup.StaticVocabularyTable(vocab_init, num_oov_buckets)

In [18]:
#Using the above table to look up the IDs of a few words:
table.lookup(tf.constant([b"This movie was faaaaaantastic".split()]))

<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[   22,    12,    11, 10053]])>

**Creating the Final Train and Test sets**

Now we will create the final training and test sets.

For creating the final training set train_set,

- we batch the reviews

- then we convert them to short sequences of words using the preprocess() function

- then encode these words using a simple encode_words() function that uses the table we just built and finally prefetch the next batch.


In [27]:
#Defining the encode_words() function to encode the words of train data using the lookup table table.
def encode_words(X_batch, y_batch):
    return table.lookup(X_batch), y_batch

In [28]:
train_set = datasets["train"].repeat().batch(32).map(preprocess)

In [30]:
train_set = train_set.map(encode_words).prefetch(1)

In [31]:
test_set = datasets["test"].batch(1000).map(preprocess)

In [32]:
test_set = test_set.map(encode_words)

In [33]:
for X_batch, y_batch in train_set.take(1):
    print(X_batch)
    print(y_batch)

tf.Tensor(
[[  22   11   28 ...    0    0    0]
 [   6   21   70 ...    0    0    0]
 [4099 6881    1 ...    0    0    0]
 ...
 [  22   12  118 ...  331 1047    0]
 [1757 4101  451 ...    0    0    0]
 [3365 4392    6 ...    0    0    0]], shape=(32, 60), dtype=int64)
tf.Tensor([0 0 0 1 1 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 1 1 1 1 1 0 0 0 1 0 0 0], shape=(32,), dtype=int64)


In [39]:
##Building the Model
embed_size = 128 #embedding size of each word.

In [40]:
model = keras.models.Sequential([
    keras.layers.Embedding(vocab_size + num_oov_buckets, embed_size,
               mask_zero=True,
               input_shape=[None]),
    keras.layers.GRU(4, return_sequences=True),
    keras.layers.GRU(2),
    keras.layers.Dense(1, activation="sigmoid")
])

In [41]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [44]:
import time

In [45]:
#measure the time of training using time module.
start = time.time()

In [46]:
model.fit(train_set, steps_per_epoch=train_size // 32, epochs=2)

Epoch 1/2
781/781 [==============================] - 88s 102ms/step - loss: 0.5910 - accuracy: 0.6626
Epoch 2/2
781/781 [==============================] - 93s 119ms/step - loss: 0.3785 - accuracy: 0.8385


In [47]:
print("Time of execution:", end-start)
model.evaluate(test_set)

NameError: name 'end' is not defined